# Task 3 – Output format study (JPEG vs PNG vs JPEG quality)

The training notebook saved `pred_A2B` / `pred_B2A` as JPEG (PIL default, quality 75).
This notebook checks whether the **file format alone** changes the evaluation score, using the
run-3 best checkpoint (`20261004_065303_generators_best.pt`) and the **same code as the class evaluation script**
(Inception-v3 IMAGENET1K_V1, `transform_input=False`, fc=Identity, Resize 299 → CenterCrop 299, ImageNet normalisation,
first 300 sorted images per folder, MiFID = mean cosine distance paired by index).

No retraining – runs on the Mac (MPS) in ~10 minutes. Run from `task3_gan/manjot/src/`.

In [1]:
import glob, json, os, re, shutil, sys
from pathlib import Path
import numpy as np, scipy.linalg, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models, torchvision.transforms as T
from PIL import Image
from scipy.spatial.distance import cosine

# work from task3_gan/manjot regardless of where the notebook is opened
HERE = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
os.chdir(HERE)
DATA = HERE.parent / "data"
CKPT = "checkpoints/20261004_065303_generators_best.pt"     # run-3 best generators
NB = "src/task3_run3.ipynb"                                  # run-3 training notebook (model code is read from it)
dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("working dir:", HERE, "| device:", dev)

working dir: ../../manjot | device: mps


## 1. Rebuild the run-3 generators from the training notebook + checkpoint

In [2]:
ns = {"torch": torch, "nn": nn, "F": F, "np": np}
cells = ["".join(c["source"]) for c in json.load(open(NB))["cells"] if c["cell_type"] == "code"]
for src in cells:
    if "class Generator" in src or "class ResBlock" in src or "class ResidualBlock" in src:
        try:
            exec(src, ns)
        except Exception as e:   # the cell also builds models with `device`; the class definitions above that line are kept
            print("  (note: model cell partly executed:", type(e).__name__, e, ")")
for src in cells:
    m = re.search(r"^G_CFG\s*=\s*dict\((.*?)\)\s*$", src, re.S | re.M)
    if m and "G_CFG" not in ns:
        exec("G_CFG = dict(" + m.group(1) + ")", ns)
assert "Generator" in ns, "could not find class Generator in the notebook"

ck = torch.load(CKPT, map_location="cpu")
cfg = ck.get("G_CFG", ns.get("G_CFG", {}))
print("G_CFG:", cfg)
G = {}
for k in ("G_AB", "G_BA"):
    G[k] = ns["Generator"](**cfg)
    G[k].load_state_dict(ck[k]); G[k].to(dev).eval()

to_in = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

@torch.no_grad()
def translate(g, path):
    x = to_in(Image.open(path).convert("RGB")).unsqueeze(0).to(dev)
    y = g(x)[0].float().cpu().clamp(-1, 1) * 0.5 + 0.5
    return Image.fromarray((y.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

photos = sorted(glob.glob(str(DATA / "photo_jpg/*.jpg")))[:300]   # only the first 300 sorted are evaluated
monets = sorted(glob.glob(str(DATA / "monet_jpg/*.jpg")))

  (note: model cell partly executed: NameError name 'device' is not defined )
G_CFG: {'ngf': 64, 'n_res': 9, 'upsample': 'convtranspose'}


## 2. Sanity check – this pipeline must reproduce the images saved by the training notebook

In [3]:
chk = translate(G["G_BA"], photos[0]); chk.save("/tmp/_chk.jpg")
a = np.asarray(Image.open("/tmp/_chk.jpg"), dtype=np.float32)
b = np.asarray(Image.open(f"outputs/pred_B2A/{Path(photos[0]).name}"), dtype=np.float32)
diff = np.abs(a - b).mean()
print(f"mean |diff| vs saved image = {diff:.2f} / 255  ->", "OK" if diff < 4 else "MISMATCH – stop here")
assert diff < 4

mean |diff| vs saved image = 3.63 / 255  -> OK


## 3. Write lossless PNG versions of the evaluated images

In [4]:
out = {"B2A": HERE / "outputs_png/pred_B2A", "A2B": HERE / "outputs_png/pred_A2B"}
for d in out.values():
    d.mkdir(parents=True, exist_ok=True)
for p in photos:
    translate(G["G_BA"], p).save(out["B2A"] / (Path(p).stem + ".png"))
for p in monets:
    translate(G["G_AB"], p).save(out["A2B"] / (Path(p).stem + ".png"))
print("wrote", len(photos), "+", len(monets), "PNGs")

wrote 300 + 300 PNGs


## 4. Class evaluation code (FID + MiFID)

In [5]:
inc = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
inc.fc = nn.Identity(); inc.to(dev).eval()
TF = T.Compose([T.Resize(299), T.CenterCrop(299), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

@torch.no_grad()
def acts(paths):
    return np.concatenate([inc(torch.stack([TF(Image.open(p).convert("RGB")) for p in paths[i:i + 32]]).to(dev)).cpu().numpy()
                           for i in range(0, len(paths), 32)])

real_m = sorted(glob.glob(str(DATA / "monet_jpg/*.jpg")))[:300]
real_p = sorted(glob.glob(str(DATA / "photo_jpg/*.jpg")))[:300]
R = {"B2A": acts(real_m), "A2B": acts(real_p)}       # real features, computed once

def score(r, paths):
    g = acts(sorted(paths)[:300])
    s1, s2 = np.cov(r, rowvar=False), np.cov(g, rowvar=False)
    c = scipy.linalg.sqrtm(s1.dot(s2)).real
    d = r.mean(0) - g.mean(0)
    return float(d.dot(d) + np.trace(s1 + s2 - 2 * c)), float(np.mean([cosine(r[i], g[i]) for i in range(len(r))]))

results = []
def report(name, b2a_paths, a2b_paths):
    (fb, mb), (fa, ma) = score(R["B2A"], b2a_paths), score(R["A2B"], a2b_paths)
    F_, M_ = (fa + fb) / 2, (ma + mb) / 2
    results.append(dict(setting=name, FID_B2A=fb, FID_A2B=fa, FID=F_, MiFID=M_, kaggle_score=-(F_ + M_) / 2))
    print(f"{name:14s}: FID_B2A={fb:.2f} FID_A2B={fa:.2f} | FID={F_:.3f} MiFID={M_:.4f} | score={-(F_ + M_) / 2:.3f}", flush=True)

## 5. JPEG (as saved by the training notebook) vs PNG

In [6]:
report("JPEG q75 (saved)", glob.glob("outputs/pred_B2A/*.jpg"), glob.glob("outputs/pred_A2B/*.jpg"))
report("PNG (lossless)", glob.glob(str(out["B2A"] / "*.png")), glob.glob(str(out["A2B"] / "*.png")))

JPEG q75 (saved): FID_B2A=101.27 FID_A2B=102.92 | FID=102.098 MiFID=0.4168 | score=-51.257


PNG (lossless): FID_B2A=105.48 FID_A2B=104.00 | FID=104.739 MiFID=0.4210 | score=-52.580


## 6. JPEG quality sweep
The real Monet/photo files are JPEGs, so Inception sees their compression texture. Here the lossless PNGs are
re-encoded at several qualities, and once with the **real files' own quantization tables**.

In [7]:
q_monet = Image.open(real_m[0]).quantization
q_photo = Image.open(real_p[0]).quantization
settings = [("q50", dict(quality=50)), ("q65", dict(quality=65)), ("q75", dict(quality=75)),
            ("q85", dict(quality=85)), ("q95", dict(quality=95)), ("real-tables", "real")]
tmp = Path("outputs_jpgtest")
for name, kw in settings:
    paths = {}
    for k in ("B2A", "A2B"):
        d = tmp / name / k; d.mkdir(parents=True, exist_ok=True)
        opts = kw if kw != "real" else dict(qtables=(q_monet if k == "B2A" else q_photo))
        for p in sorted(glob.glob(str(out[k] / "*.png"))):
            Image.open(p).save(d / (Path(p).stem + ".jpg"), **opts)
        paths[k] = glob.glob(str(d / "*.jpg"))
    report(name, paths["B2A"], paths["A2B"])
shutil.rmtree(tmp)

q50           : FID_B2A=98.57 FID_A2B=103.71 | FID=101.143 MiFID=0.4148 | score=-50.779


q65           : FID_B2A=98.83 FID_A2B=102.80 | FID=100.819 MiFID=0.4150 | score=-50.617


q75           : FID_B2A=99.04 FID_A2B=102.75 | FID=100.896 MiFID=0.4147 | score=-50.655


q85           : FID_B2A=99.84 FID_A2B=102.86 | FID=101.352 MiFID=0.4159 | score=-50.884


q95           : FID_B2A=101.47 FID_A2B=103.02 | FID=102.246 MiFID=0.4171 | score=-51.331


real-tables   : FID_B2A=99.04 FID_A2B=102.75 | FID=100.896 MiFID=0.4147 | score=-50.655


## 7. Summary

In [8]:
import pandas as pd
df = pd.DataFrame(results).sort_values("kaggle_score", ascending=False)
df.to_csv("output_format_study.csv", index=False)
df

,setting,FID_B2A,FID_A2B,FID,MiFID,kaggle_score
3,q65,98.833354,102.803982,100.818668,0.415003,-50.616836
4,q75,99.041297,102.749966,100.895632,0.414712,-50.655172
7,real-tables,99.041297,102.749966,100.895632,0.414712,-50.655172
2,q50,98.571447,103.713765,101.142606,0.414824,-50.778715
5,q85,99.841704,102.861716,101.351710,0.415902,-50.883806
0,JPEG q75 (saved),101.272245,102.924104,102.098175,0.416815,-51.257495
6,q95,101.473922,103.017532,102.245727,0.417052,-51.331389
1,PNG (lossless),105.476894,104.001327,104.739111,0.421002,-52.580056
